# Batch Normalization

**Goal:** Implement BatchNorm (1D) from scratch in PyTorch — training mode with batch statistics and affine transform, running statistics for eval mode, and an optional manual backward pass. Validate all outputs against `torch.nn.BatchNorm1d`, then demonstrate how BN stabilizes activations across deep layers.

Cross-links: `[[vanishing-exploding-gradients]]`, `[[activations-tanh-relu]]`, `[[broadcasting-in-nns]]`

## Configuration

Device, random seed, and default dtype come from `shared.config.configure()`. This reads `config.toml` at the repo root and applies the chosen device / seed / dtype for the session.

In [1]:
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # noqa
import matplotlib.pyplot as plt  # noqa: E402
import torch  # noqa: E402
import torch.nn.functional as F  # noqa: E402


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

from shared.config import configure  # noqa: E402

device = configure()
print("running on:", device)

running on: mps


## From Scratch: BatchNorm1d Forward Pass

For a mini-batch `x` of shape `(B, D)`:

```text
mu_B  = (1/B) sum_i x_i           # batch mean, shape (D,)
var_B = (1/B) sum_i (x_i - mu_B)^2  # biased batch variance, shape (D,)
xhat  = (x - mu_B) / sqrt(var_B + eps)   # normalized
y     = gamma * xhat + beta        # affine transform
```

**Running statistics** (updated only during training, used during eval):

```text
running_mean <- momentum * running_mean + (1 - momentum) * mu_B
running_var  <- momentum * running_var  + (1 - momentum) * unbiased_var_B
```

Note: PyTorch uses *unbiased* variance (Bessel's correction) for the running stats, but *biased* variance for the normalization step — this matches the original paper.

`gamma` and `beta` each have shape `(D,)` and are broadcast over the batch axis.

In [2]:
import torch.nn as nn
from dataclasses import dataclass


@dataclass
class BNState:
    """Mutable running statistics (held separately to keep forward pure)."""
    running_mean: torch.Tensor
    running_var: torch.Tensor


def batchnorm1d_forward(
    x: torch.Tensor,
    gamma: torch.Tensor,
    beta: torch.Tensor,
    state: BNState,
    training: bool = True,
    momentum: float = 0.1,
    eps: float = 1e-5,
) -> torch.Tensor:
    """BatchNorm1d forward pass from scratch.

    Args:
        x: Input of shape (B, D).
        gamma: Scale parameter of shape (D,).
        beta: Shift parameter of shape (D,).
        state: Running statistics (updated in-place during training).
        training: If True use batch statistics; if False use running statistics.
        momentum: EMA factor for running stats (matches PyTorch convention).
        eps: Numerical stability term.

    Returns:
        Normalised and affine-transformed tensor of shape (B, D).
    """
    if training:
        # Biased batch variance (denominator = B) — used for normalisation
        mu = x.mean(dim=0)           # (D,)
        var_biased = x.var(dim=0, unbiased=False)   # (D,)
        xhat = (x - mu) / torch.sqrt(var_biased + eps)

        # Running stats use unbiased variance for consistency with PyTorch
        var_unbiased = x.var(dim=0, unbiased=True)  # (D,)
        # Update running statistics (EMA)
        new_running_mean = (1 - momentum) * state.running_mean + momentum * mu
        new_running_var  = (1 - momentum) * state.running_var  + momentum * var_unbiased
        state.running_mean = new_running_mean.detach()
        state.running_var  = new_running_var.detach()
    else:
        # Eval: use accumulated running statistics
        xhat = (x - state.running_mean) / torch.sqrt(state.running_var + eps)

    return gamma * xhat + beta


# Quick sanity check on a tiny batch
torch.manual_seed(42)
B, D = 4, 3
x_small = torch.randn(B, D, device=device)
gamma_ones = torch.ones(D, device=device)
beta_zeros = torch.zeros(D, device=device)
state_small = BNState(
    running_mean=torch.zeros(D, device=device),
    running_var=torch.ones(D, device=device),
)

out_scratch = batchnorm1d_forward(x_small, gamma_ones, beta_zeros, state_small)
print("Input:\n", x_small)
print("\nBN output (gamma=1, beta=0):\n", out_scratch)
print("\nMean per feature (should be ~0):", out_scratch.mean(dim=0))
print("Var  per feature (should be ~1):", out_scratch.var(dim=0, unbiased=False))

Input:
 tensor([[ 0.9047,  0.2227,  0.1460],
        [ 0.7360, -0.4142, -0.1106],
        [ 0.4540, -3.3259, -0.2307],
        [-0.9187,  0.2464,  1.0246]], device='mps:0')

BN output (gamma=1, beta=0):
 tensor([[ 0.8500,  0.7067, -0.1248],
        [ 0.6153,  0.2741, -0.6474],
        [ 0.2227, -1.7038, -0.8920],
        [-1.6880,  0.7229,  1.6642]], device='mps:0')

Mean per feature (should be ~0): tensor([2.9802e-08, 2.9802e-08, 2.9802e-08], device='mps:0')
Var  per feature (should be ~1): tensor([1.0000, 1.0000, 1.0000], device='mps:0')


## Validation: Compare to `nn.BatchNorm1d`

We copy our learned `gamma` and `beta` into a fresh `nn.BatchNorm1d` module and assert that the outputs match to within floating-point precision. Key things to match:
- Same `eps` and `momentum`.
- `nn.BatchNorm1d` initialises `weight=1, bias=0`; we use the same.
- PyTorch uses biased variance internally for the normalization step — our scratch implementation does the same.

In [3]:
torch.manual_seed(0)
B, D = 32, 8
x_val = torch.randn(B, D, device=device)

EPS = 1e-5
MOMENTUM = 0.1

# --- Our scratch implementation ---
gamma_v = torch.ones(D, device=device)
beta_v  = torch.zeros(D, device=device)
state_v = BNState(
    running_mean=torch.zeros(D, device=device),
    running_var=torch.ones(D, device=device),
)
out_scratch_v = batchnorm1d_forward(
    x_val, gamma_v, beta_v, state_v, training=True,
    momentum=MOMENTUM, eps=EPS,
)

# --- PyTorch nn.BatchNorm1d ---
bn_ref = nn.BatchNorm1d(D, eps=EPS, momentum=MOMENTUM).to(device)
bn_ref.weight.data.fill_(1.0)   # gamma = 1
bn_ref.bias.data.fill_(0.0)     # beta  = 0
bn_ref.train()
out_torch = bn_ref(x_val)

max_diff = (out_scratch_v - out_torch).abs().max().item()
print(f"Max absolute difference (training mode): {max_diff:.2e}")
assert max_diff < 1e-5, f"Training-mode outputs diverge: {max_diff}"
print("PASSED: scratch matches nn.BatchNorm1d in training mode")

# --- Running-stats check ---
print("\nRunning mean (scratch):", state_v.running_mean[:4].tolist())
print("Running mean (torch):  ", bn_ref.running_mean[:4].tolist())
rstat_diff = (state_v.running_mean - bn_ref.running_mean).abs().max().item()
assert rstat_diff < 1e-5, f"Running mean mismatch: {rstat_diff}"
print("PASSED: running means match")

rvar_diff = (state_v.running_var - bn_ref.running_var).abs().max().item()
assert rvar_diff < 1e-5, f"Running var mismatch: {rvar_diff}"
print("PASSED: running vars match")

Max absolute difference (training mode): 2.38e-07
PASSED: scratch matches nn.BatchNorm1d in training mode

Running mean (scratch): [-0.01612640731036663, -0.021458998322486877, 0.003343591932207346, -0.02582578733563423]
Running mean (torch):   [-0.01612640731036663, -0.021458998322486877, 0.003343591932207346, -0.02582578733563423]
PASSED: running means match
PASSED: running vars match


## Eval Mode: Using Running Statistics

After training, `BatchNorm1d` switches to using `running_mean` and `running_var` instead of batch statistics. This makes inference independent of the batch size and fully deterministic. We demonstrate this by running several training steps to warm up the running stats, then switching to eval mode.

In [4]:
torch.manual_seed(1)
B_train, D_eval = 64, 4

# Warm up running statistics over 20 mini-batches
state_eval = BNState(
    running_mean=torch.zeros(D_eval, device=device),
    running_var=torch.ones(D_eval, device=device),
)
gamma_e = torch.ones(D_eval, device=device)
beta_e  = torch.zeros(D_eval, device=device)
bn_eval_ref = nn.BatchNorm1d(D_eval, eps=1e-5, momentum=0.1).to(device)
bn_eval_ref.train()

for step in range(20):
    x_batch = torch.randn(B_train, D_eval, device=device) + 2.0  # shift mean to 2
    _ = batchnorm1d_forward(x_batch, gamma_e, beta_e, state_eval, training=True)
    _ = bn_eval_ref(x_batch)

print("After 20 training steps:")
print(f"  Running mean (scratch):  {state_eval.running_mean.tolist()}")
print(f"  Running mean (torch):    {bn_eval_ref.running_mean.tolist()}")

# Now evaluate on a single sample (batch size 1 — only valid in eval mode)
x_single = torch.randn(1, D_eval, device=device) + 2.0

out_eval_scratch = batchnorm1d_forward(
    x_single, gamma_e, beta_e, state_eval, training=False
)
bn_eval_ref.eval()
out_eval_torch = bn_eval_ref(x_single)

max_diff_eval = (out_eval_scratch - out_eval_torch).abs().max().item()
print(f"\nMax absolute difference (eval mode, batch=1): {max_diff_eval:.2e}")
assert max_diff_eval < 1e-5, f"Eval-mode outputs diverge: {max_diff_eval}"
print("PASSED: eval-mode scratch matches nn.BatchNorm1d (batch size 1 works only in eval)")

# Demonstrate instability in training mode with batch size 1
try:
    bn_eval_ref.train()
    out_train_bs1 = bn_eval_ref(x_single)
    print("\nWarning: train mode with batch=1 ran but produces degenerate statistics")
    print("  (var is 0, output is nan or arbitrary when eps is tiny)")
except Exception as exc:
    print(f"\nExpected error with batch=1 in train mode: {exc}")

After 20 training steps:
  Running mean (scratch):  [1.7504408359527588, 1.738876223564148, 1.7558236122131348, 1.7906646728515625]
  Running mean (torch):    [1.7504408359527588, 1.738876223564148, 1.7558236122131348, 1.7906646728515625]

Max absolute difference (eval mode, batch=1): 5.96e-08
PASSED: eval-mode scratch matches nn.BatchNorm1d (batch size 1 works only in eval)

Expected error with batch=1 in train mode: Expected more than 1 value per channel when training, got input size torch.Size([1, 4])


## Optional: Manual Backward Pass vs. Autograd

The BN backward computes gradients for `gamma`, `beta`, and the input `x`. We implement it from first principles following Ioffe & Szegedy (2015), then verify against `torch.autograd.grad`.

```text
dx_hat  = dy * gamma                          # (B, D)
dvar    = sum_i dx_hat_i * (x_i - mu) * -0.5 * (var + eps)^{-3/2}
dmu     = sum_i dx_hat_i * -1/sqrt(var+eps) + dvar * sum_i -2(x_i-mu)/B
dx_i    = dx_hat_i / sqrt(var+eps) + dvar * 2(x_i-mu)/B + dmu/B
dgamma  = sum_i dy_i * xhat_i
dbeta   = sum_i dy_i
```

In [5]:
def batchnorm1d_backward(
    dy: torch.Tensor,
    x: torch.Tensor,
    gamma: torch.Tensor,
    eps: float = 1e-5,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Manual BN backward for (B, D) inputs.

    Returns:
        dx: gradient w.r.t. input x, shape (B, D)
        dgamma: gradient w.r.t. gamma, shape (D,)
        dbeta: gradient w.r.t. beta, shape (D,)
    """
    B = x.shape[0]
    mu  = x.mean(dim=0)                          # (D,)
    var = x.var(dim=0, unbiased=False)           # (D,)
    xhat = (x - mu) / torch.sqrt(var + eps)      # (B, D)

    dgamma = (dy * xhat).sum(dim=0)              # (D,)
    dbeta  = dy.sum(dim=0)                       # (D,)

    dxhat  = dy * gamma                          # (B, D)
    dvar   = (dxhat * (x - mu) * (-0.5) * (var + eps).pow(-1.5)).sum(dim=0)  # (D,)
    dmu    = (dxhat * (-1.0 / torch.sqrt(var + eps))).sum(dim=0) + dvar * (-2.0 * (x - mu)).sum(dim=0) / B
    dx     = dxhat / torch.sqrt(var + eps) + dvar * 2.0 * (x - mu) / B + dmu / B

    return dx, dgamma, dbeta


# Validation against autograd
torch.manual_seed(7)
B_bw, D_bw = 16, 5
x_bw = torch.randn(B_bw, D_bw, device=device, requires_grad=True)
gamma_bw = torch.ones(D_bw, device=device, requires_grad=True)
beta_bw  = torch.zeros(D_bw, device=device, requires_grad=True)
dy_bw    = torch.randn(B_bw, D_bw, device=device)

# Forward (autograd tracked)
mu_bw   = x_bw.mean(dim=0)
var_bw  = x_bw.var(dim=0, unbiased=False)
xhat_bw = (x_bw - mu_bw) / torch.sqrt(var_bw + 1e-5)
out_bw  = gamma_bw * xhat_bw + beta_bw
out_bw.backward(dy_bw)

dx_auto     = x_bw.grad.clone()
dgamma_auto = gamma_bw.grad.clone()
dbeta_auto  = beta_bw.grad.clone()

# Manual backward (on detached tensors)
dx_man, dgamma_man, dbeta_man = batchnorm1d_backward(
    dy_bw, x_bw.detach(), gamma_bw.detach(), eps=1e-5
)

print(f"dx     max diff: {(dx_man - dx_auto).abs().max().item():.2e}")
print(f"dgamma max diff: {(dgamma_man - dgamma_auto).abs().max().item():.2e}")
print(f"dbeta  max diff: {(dbeta_man - dbeta_auto).abs().max().item():.2e}")

assert (dx_man - dx_auto).abs().max() < 1e-4, "dx mismatch"
assert (dgamma_man - dgamma_auto).abs().max() < 1e-4, "dgamma mismatch"
assert (dbeta_man - dbeta_auto).abs().max() < 1e-4, "dbeta mismatch"
print("PASSED: manual BN backward matches autograd")

dx     max diff: 2.38e-07
dgamma max diff: 0.00e+00
dbeta  max diff: 0.00e+00
PASSED: manual BN backward matches autograd


## Idiomatic PyTorch: `nn.BatchNorm1d`

In production, always use `torch.nn.BatchNorm1d` (or `BatchNorm2d` for CNNs). The key operational requirement: **call `model.eval()` before inference**. Forgetting this is one of the most common BN bugs in deployed models.

In [6]:
# Standard idiom
class SmallMLP(nn.Module):
    def __init__(self, in_features: int, hidden: int, out_features: int, use_bn: bool = True):
        super().__init__()
        self.use_bn = use_bn
        self.fc1 = nn.Linear(in_features, hidden)
        self.bn1 = nn.BatchNorm1d(hidden) if use_bn else nn.Identity()
        self.fc2 = nn.Linear(hidden, hidden)
        self.bn2 = nn.BatchNorm1d(hidden) if use_bn else nn.Identity()
        self.fc3 = nn.Linear(hidden, out_features)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.bn1(self.fc1(x)))
        x = F.relu(self.bn2(self.fc2(x)))
        return self.fc3(x)


torch.manual_seed(42)
model_bn  = SmallMLP(16, 64, 4, use_bn=True).to(device)
model_no_bn = SmallMLP(16, 64, 4, use_bn=False).to(device)

# Training loop — compare activation norms
def train_one_epoch(model: nn.Module, steps: int = 50) -> list[float]:
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    model.train()
    losses = []
    for _ in range(steps):
        x = torch.randn(64, 16, device=device)
        y = (x[:, :4].sum(dim=1) > 0).long()
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(loss.item())
    return losses

losses_bn   = train_one_epoch(model_bn)
losses_nobn = train_one_epoch(model_no_bn)

# Plot
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(losses_bn,   label="With BatchNorm")
ax.plot(losses_nobn, label="Without BatchNorm")
ax.set_xlabel("Step")
ax.set_ylabel("Cross-entropy loss")
ax.set_title("Effect of BatchNorm on Training Loss")
ax.legend()
fig.tight_layout()
plt.savefig("bn_training_loss.png", dpi=80)
plt.close(fig)
print("Saved bn_training_loss.png")

print(f"\nFinal loss (BN):    {losses_bn[-1]:.4f}")
print(f"Final loss (no BN): {losses_nobn[-1]:.4f}")

# Eval mode matters
model_bn.eval()
x_test = torch.randn(128, 16, device=device)
with torch.no_grad():
    logits_eval = model_bn(x_test)
print(f"\nEval-mode inference output shape: {logits_eval.shape}  (no crash with batch=128)")

# Demonstrate train-mode eval issue
model_bn.train()   # accidentally left in training mode
with torch.no_grad():
    logits_train_mode = model_bn(x_test)
diff = (logits_eval - logits_train_mode).abs().max().item()
print(f"Max output diff (eval vs train mode during inference): {diff:.4f}")
print("(Non-zero diff shows BN train/eval mode affects inference outputs)")

Saved bn_training_loss.png

Final loss (BN):    0.3753
Final loss (no BN): 0.5541

Eval-mode inference output shape: torch.Size([128, 4])  (no crash with batch=128)
Max output diff (eval vs train mode during inference): 0.3544
(Non-zero diff shows BN train/eval mode affects inference outputs)


## Demonstrating Activation Stabilization

Without BatchNorm, activations in deep networks can grow or shrink exponentially as signals pass through many layers — a symptom related to `[[vanishing-exploding-gradients]]`. BatchNorm anchors each layer's input distribution, keeping the scale stable regardless of depth.

In [7]:
def measure_activation_norms(use_bn: bool, depth: int = 8, width: int = 128) -> list[float]:
    """Build a deep network and record mean activation norm per layer."""
    layers_no_bn = []
    for _ in range(depth):
        layers_no_bn.append(nn.Linear(width, width))
        layers_no_bn.append(nn.ReLU())
        if use_bn:
            layers_no_bn.append(nn.BatchNorm1d(width))

    model = nn.Sequential(*layers_no_bn).to(device)
    # Deliberately large init to provoke instability
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=1.5)

    x = torch.randn(64, width, device=device)
    norms: list[float] = []
    with torch.no_grad():
        h = x
        for layer in model:
            h = layer(h)
            if isinstance(layer, nn.ReLU):
                norms.append(h.norm(dim=1).mean().item())
    return norms


norms_with_bn    = measure_activation_norms(use_bn=True)
norms_without_bn = measure_activation_norms(use_bn=False)

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(norms_with_bn,    marker="o", label="With BatchNorm")
ax.plot(norms_without_bn, marker="s", label="Without BatchNorm")
ax.set_xlabel("Layer index (post-ReLU)")
ax.set_ylabel("Mean activation L2 norm")
ax.set_title("Activation Norms Through Depth (large init std=1.5)")
ax.legend()
fig.tight_layout()
plt.savefig("bn_activation_norms.png", dpi=80)
plt.close(fig)
print("Saved bn_activation_norms.png")

for i, (with_bn, without_bn) in enumerate(zip(norms_with_bn, norms_without_bn)):
    print(f"  Layer {i}: norm_BN={with_bn:7.2f}  norm_no_BN={without_bn:12.4f}")

Saved bn_activation_norms.png
  Layer 0: norm_BN= 135.65  norm_no_BN=    133.2218
  Layer 1: norm_BN= 136.74  norm_no_BN=   1728.6465
  Layer 2: norm_BN= 135.28  norm_no_BN=  21110.8984
  Layer 3: norm_BN= 135.57  norm_no_BN= 260683.5938
  Layer 4: norm_BN= 131.06  norm_no_BN=3117699.5000
  Layer 5: norm_BN= 132.63  norm_no_BN=40831724.0000
  Layer 6: norm_BN= 133.57  norm_no_BN=561079552.0000
  Layer 7: norm_BN= 133.43  norm_no_BN=6453677056.0000


## Takeaways

- **Training mode** normalizes with *batch* mean and *biased* variance; updates running stats using EMA with *unbiased* variance.
- **Eval mode** uses the accumulated `running_mean` / `running_var` — forgetting to call `model.eval()` is a silent, damaging bug.
- **Affine parameters** `gamma` and `beta` restore representational power: without them, every layer's output is forced to zero mean and unit variance regardless of what the network learned.
- **Backward pass** is non-trivial because `mu` and `var` are functions of the entire batch — every sample sees gradients from every other sample in the batch.
- **Stabilizes activations:** keeps signal scale bounded through depth, mitigating `[[vanishing-exploding-gradients]]` even with large weight initializations.
- **Avoid for small batches:** with B < 8 the batch statistics are noisy; prefer Layer Norm or Group Norm.
- **Bias before BN is redundant:** a linear layer bias `b` is absorbed by BN's mean-centering and then replaced by the learned `beta` — use `bias=False` in linear/conv layers immediately followed by BN.